# Statistical Outputs of Socio-Economic, Institutional, and Experiential Variables on Community Knowledge Management Ratings in Northern Kenya Pastoral Rangelands Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, focusing on statistical outputs for community knowledge management ratings in pastoral rangeland management.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.3mhk-e0vy/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.3mhk-e0vy/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("License:", metadata.license)
print("Citation / CiteAs:", getattr(metadata, 'citeAs', None))
print("Spatial Coverage:", getattr(metadata, 'spatialCoverage', None))
print("Temporal Coverage:", getattr(metadata, 'temporalCoverage', None))
print("Date Published:", getattr(metadata, 'datePublished', None))


## 2. Data Overview
Review available record sets, fields, and their `@id`. Every entity is referenced by its unique `@id` as per Croissant semantics.

Let's enumerate the available record sets and their IDs:

In [ ]:
# The Croissant metadata exposes recordSets as metadata.recordSet (a list of objects)
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No RecordSets found in this dataset's schema.")
else:
    print("RecordSets (@id):")
    for rs in record_sets:
        print("  -", rs['@id'])

    # Print available fields and columns for each record set
    for rs in record_sets:
        print(f"\nFields and columns for RecordSet {rs['@id']}:")
        fields = rs.get('field', [])
        if fields:
            for f in fields:
                print(f"    Field @id: {f['@id']}, name: {f.get('name', '')}, dataType: {f.get('dataType', '')}")
        columns = rs.get('column', [])
        if columns:
            for c in columns:
                print(f"    Column @id: {c['@id']}, name: {c.get('name', '')}, dataType: {c.get('dataType', '')}")


## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Each record set is referenced by its `@id`, and DataFrames are keyed by those IDs.


In [ ]:
# Extract data from each record set -- all entities referenced by @id
dataframes = {}
rs_ids = []

if not record_sets:
    print("No RecordSets defined in metadata.")
else:
    # List of recordSet @id
    rs_ids = [rs['@id'] for rs in record_sets]

    print("Attempting to load data for each RecordSet...\n")
    for record_set_id in rs_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded RecordSet '{record_set_id}' with shape {df.shape}")
        except Exception as e:
            print(f"Could not load RecordSet '{record_set_id}':", e)

    # For demonstration, print columns of the first populated DataFrame
    for rs_id, df in dataframes.items():
        print(f"\nColumns for {rs_id}:", df.columns.tolist())
        print(df.head())
        break


## 4. Exploratory Data Analysis (EDA)
Let's apply typical EDA steps: filtering numeric records, normalizing, and grouping by categorical attributes.

All operations reference entity and column `@id`s for reproducibility and semantic consistency.

In [ ]:
# Find a numeric column and a grouping field by @id
import numpy as np

# Choose first available dataframe
if not dataframes:
    print("No populated DataFrames found.\nAbort EDA.")
else:
    df_id = list(dataframes.keys())[0]
    df = dataframes[df_id]

    # Try to identify numeric columns
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        # Choose any threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to identify a grouping field (categorical type)
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_fields:
            # Take first group field
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No categorical grouping field found.")
    else:
        print("No numeric columns found in DataFrame.")


## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib and seaborn.

All fields and axes are referenced by their `@id`.

In [ ]:
if not dataframes:
    print("No DataFrames to visualize.")
else:
    df_id = list(dataframes.keys())[0]
    df = dataframes[df_id]

    # Find numeric and categorical columns
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if numeric_cols and cat_cols:
        x_id = cat_cols[0]
        y_id = numeric_cols[0]
        plt.figure(figsize=(8,5))
        sns.boxplot(x=x_id, y=y_id, data=df)
        plt.title(f"Distribution of {y_id} by {x_id}")
        plt.xlabel(x_id)
        plt.ylabel(y_id)
        plt.show()

        plt.figure(figsize=(8,5))
        sns.histplot(df[y_id], bins=20, kde=True)
        plt.title(f"Histogram of {y_id}")
        plt.xlabel(y_id)
        plt.ylabel("Frequency")
        plt.show()
    elif numeric_cols:
        y_id = numeric_cols[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[y_id], bins=20, kde=True)
        plt.title(f"Histogram of {y_id}")
        plt.xlabel(y_id)
        plt.ylabel("Frequency")
        plt.show()
    else:
        print("No numeric/categorical columns to visualize.")


## 6. Conclusion
This notebook has demonstrated loading, overview, extraction, EDA, and visualization for the FAIR² dataset package using `mlcroissant`. All entities (record sets, fields, columns) are referenced exclusively by their `@id` for semantic clarity. 

Key observations:
- Metadata provides extensive description, spatial and temporal coverage, and explicit data limitations and biases.
- Data is organized into record sets as defined by Croissant schema.
- Analysis can leverage numeric and categorical field `@id`s for reproducible exploration.
- Visualizations highlight relationships between fields and can be extended using `mlcroissant`'s metadata access.

For advanced use, see [mlcroissant documentation](https://github.com/mlcommons/croissant) or further Croissant schema details.